# 重子声波振荡 (BAO) 似然

本教程演示 HIcosmo 的 BAO 似然函数。

**关键 API**：
- `BAO_likelihood(model, dataset)` — 创建 BAO 似然
- `available_bao_datasets()` — 列出可用数据集
- `omega_b_mode='free'|'bbn_prior'|'fixed'` — Ω_b 处理方式
- `hicosmo('LCDM', 'bao', ...)` — 极简推断 API

In [ ]:
import hicosmo as hc
hc.init()

## 1. 可用数据集

In [ ]:
from hicosmo.likelihoods import available_bao_datasets

print("可用 BAO 数据集:", available_bao_datasets())

## 2. BAO 物理量

- **横向**：$D_M(z)/r_d$ — 角直径距离与声学视界之比
- **径向**：$D_H(z)/r_d = c/(H(z) \cdot r_d)$ — Hubble 距离与声学视界之比

In [ ]:
from hicosmo.models import LCDM

lcdm = LCDM(H0=67.36, Omega_m=0.3153, Omega_b=0.0493)
rd = lcdm.sound_horizon_drag()

print(f"声学视界 r_d = {rd:.2f} Mpc")
print(f"\nz=0.5: D_M/r_d = {lcdm.comoving_distance(0.5)/rd:.3f}")
print(f"z=0.5: D_H/r_d = {299792.458/(lcdm.E_z(0.5)*lcdm.params['H0'])/rd:.3f}")

## 3. 创建似然函数

In [ ]:
from hicosmo.likelihoods import BAO_likelihood

# 创建 DESI 2024 BAO 似然
bao = BAO_likelihood(LCDM, "desi2024")

print(f"数据集: {bao.dataset_name}")
print(f"nuisance: {[p.name for p in bao.nuisance_parameters]}")

In [ ]:
# 计算似然值
print(f"log L(Planck) = {bao(H0=67.36, Omega_m=0.3153):.2f}")
print(f"log L(H0=70)  = {bao(H0=70, Omega_m=0.3):.2f}")

## 4. omega_b_mode 处理方式

BAO 约束 $D_M/r_d$ 和 $D_H/r_d$，而 $r_d$ 依赖于 $\Omega_b$。三种处理方式：

In [ ]:
# 'free': H0_rd 作为 nuisance（默认）
bao_free = BAO_likelihood(LCDM, "desi2024", omega_b_mode='free')
print(f"free:      nuisance = {[p.name for p in bao_free.nuisance_parameters]}")

# 'bbn_prior': 添加 BBN 先验
bao_bbn = BAO_likelihood(LCDM, "desi2024", omega_b_mode='bbn_prior')
print(f"bbn_prior: nuisance = {[p.name for p in bao_bbn.nuisance_parameters]}")

# 'fixed': 固定 Planck 值
bao_fixed = BAO_likelihood(LCDM, "desi2024", omega_b_mode='fixed')
print(f"fixed:     nuisance = {[p.name for p in bao_fixed.nuisance_parameters]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 比较不同 omega_b_mode
H0_grid = np.linspace(60, 80, 50)
L_free = np.array([bao_free(H0=H0, Omega_m=0.3) for H0 in H0_grid])
L_bbn = np.array([bao_bbn(H0=H0, Omega_m=0.3) for H0 in H0_grid])
L_fixed = np.array([bao_fixed(H0=H0, Omega_m=0.3) for H0 in H0_grid])

plt.figure(figsize=(10, 5))
plt.plot(H0_grid, np.exp(L_free - L_free.max()), 'b-', lw=2, label="'free'")
plt.plot(H0_grid, np.exp(L_bbn - L_bbn.max()), 'r-', lw=2, label="'bbn_prior'")
plt.plot(H0_grid, np.exp(L_fixed - L_fixed.max()), 'g--', lw=2, label="'fixed'")
plt.xlabel(r'$H_0$ [km/s/Mpc]'); plt.ylabel('Likelihood')
plt.title('omega_b_mode 效果对比'); plt.legend()
plt.savefig('figures/04_bao_omega_b_modes.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 5. DESI 2024 数据

In [ ]:
# BAO 距离-红移关系
z_th = np.linspace(0.1, 2.5, 100)
DM_rd = [lcdm.comoving_distance(z)/rd for z in z_th]
DH_rd = [299792.458/(lcdm.E_z(z)*lcdm.params['H0'])/rd for z in z_th]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z_th, DM_rd, 'b-', lw=2); axes[0].set_xlabel('z'); axes[0].set_ylabel(r'$D_M/r_d$'); axes[0].set_title('横向 BAO')
axes[1].plot(z_th, DH_rd, 'r-', lw=2); axes[1].set_xlabel('z'); axes[1].set_ylabel(r'$D_H/r_d$'); axes[1].set_title('径向 BAO')
plt.tight_layout()
plt.savefig('figures/04_bao_theory.pdf', dpi=150, bbox_inches='tight')
plt.show()

## 6. MCMC 参数约束

In [ ]:
from hicosmo import hicosmo

# 极简 API
inf = hicosmo('LCDM', 'bao', ['H0', 'Omega_m'])
samples = inf.run(num_samples=2000)
inf.summary()

In [ ]:
inf.corner_plot('figures/04_bao_corner.pdf')

## 7. 联合约束（SN + BAO）

In [ ]:
# 极简 API 支持多似然
inf_joint = hicosmo('LCDM', ['sn', 'bao'], ['H0', 'Omega_m'])
samples = inf_joint.run(num_samples=2000)
inf_joint.summary()

In [ ]:
inf_joint.corner_plot('figures/04_bao_sn_joint.pdf')

## 8. 对象 API（更多控制）

In [ ]:
from hicosmo.samplers import MCMC
from hicosmo.likelihoods import SN_likelihood, BAO_likelihood
from hicosmo.visualization import Plotter

# 自定义似然组合
sne = SN_likelihood(LCDM, "pantheon+")
bao = BAO_likelihood(LCDM, "desi2024", omega_b_mode='bbn_prior')

params = {
    'H0': (70.0, 60.0, 80.0),
    'Omega_m': (0.3, 0.1, 0.5),
}

# 联合似然用 + 运算符
mcmc = MCMC(params, sne + bao, chain_name='bao_custom')
mcmc.run(num_samples=2000)
mcmc.print_summary()

In [ ]:
plotter = Plotter('bao_custom')
plotter.corner(['H0', 'Omega_m'])
plotter.report()

## 9. 多链对比

In [ ]:
# 分别运行 SN、BAO、联合
mcmc_sn = MCMC(params, sne, chain_name='tut_sn')
mcmc_sn.run(num_samples=2000)

mcmc_bao = MCMC(params, bao, chain_name='tut_bao')
mcmc_bao.run(num_samples=2000)

mcmc_joint = MCMC(params, sne + bao, chain_name='tut_joint')
mcmc_joint.run(num_samples=2000)

In [ ]:
# 多链角图
plotter = Plotter(['tut_sn', 'tut_bao', 'tut_joint'], labels=['SN', 'BAO', 'SN+BAO'])
plotter.corner(['H0', 'Omega_m'], filename='figures/04_multi_chain_corner.pdf')

## API 速查

```python
from hicosmo import hicosmo, list_likelihoods
from hicosmo.likelihoods import BAO_likelihood, available_bao_datasets

# 可用数据集
available_bao_datasets()  # ['desi2024', 'sdss_dr12', 'sdss_dr16', 'boss_dr12', 'sixdf']

# 创建似然
bao = BAO_likelihood(LCDM, "desi2024")                      # 默认
bao = BAO_likelihood(LCDM, "desi2024", omega_b_mode='free') # H0_rd 作为 nuisance
bao = BAO_likelihood(LCDM, "desi2024", omega_b_mode='bbn_prior')  # BBN 先验
bao = BAO_likelihood(LCDM, "desi2024", omega_b_mode='fixed')     # 固定 Planck 值

# 计算似然
log_L = bao(H0=70, Omega_m=0.3)

# 访问属性
bao.dataset_name          # 数据集名称
bao.omega_b_mode          # Ω_b 处理方式
bao.nuisance_parameters   # nuisance 参数列表

# 似然组合
joint = sne + bao         # 用 + 运算符

# 极简推断 API
inf = hicosmo('LCDM', 'bao', ['H0', 'Omega_m'])       # BAO only
inf = hicosmo('LCDM', ['sn', 'bao'], ['H0', 'Omega_m'])  # 联合
inf.run(num_samples=5000)
inf.summary()
inf.corner_plot('output.pdf')
```